In [16]:
import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
import plotly.express as px
from common_lib.metrics import aggregate_metric, aggregate_segment_metric, plot_metric

## AB test assign

In [17]:
query_location = './sql/abtest_assign.sql'
parameters = {
    'experiment_name': 'Temp_Generator',
    'assignmentstartdate': '2026-08-05'
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 9.7 MB when run.
Estimated query cost: $0.00


In [18]:
refresh_data = True

In [19]:
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query='./sql/abtest_assign.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/abtest_assign_data.pkl')
else:
    data = pd.read_pickle('./data/abtest_assign_data.pkl')

In [20]:
data

,user_id,experiment_name,exposure_dt,assigned_dt,min_start_dt,end_dt,variant
0,3B34729DD8206966,Temp_Generator,2026-08-22,2026-08-22,2026-08-05,2026-08-27,Test
1,83EE12A4573B6F68,Temp_Generator,2026-08-22,2026-08-22,2026-08-05,2026-08-27,Test
2,58FE19278D361572,Temp_Generator,2026-08-22,2026-08-22,2026-08-05,2026-08-27,Test
3,FCCE8431FFFF6ACA,Temp_Generator,2026-08-22,2026-08-22,2026-08-05,2026-08-27,Test
4,883933050DA6EB77,Temp_Generator,2026-08-22,2026-08-22,2026-08-05,2026-08-27,Test
...,...,...,...,...,...,...,...
112879,595792F20CA2993B,Temp_Generator,2026-08-18,2026-08-07,2026-08-05,2026-08-27,control
112880,7F92A3BF46F5B893,Temp_Generator,NaT,2026-08-07,2026-08-05,2026-08-27,control
112881,6D5C5CB976AABEDF,Temp_Generator,2026-08-17,2026-08-07,2026-08-05,2026-08-27,control
112882,3FEE8F217AA48BB,Temp_Generator,NaT,2026-08-07,2026-08-05,2026-08-27,control


## ALL metrics

In [21]:
query_location = './sql/abtest_metrics.sql'
parameters = {
    'lookback':28,
    'assignmentstartdate': '2026-08-05'
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 3.03 GB when run.
Estimated query cost: $0.02


In [22]:
if refresh_data:
    data_metrics = bqc.get(query='./sql/abtest_metrics.sql', is_path=True, query_parameters=parameters)
    data_metrics.to_pickle('./data/abtest_metrics.pkl')
else:
    data_metrics = pd.read_pickle('./data/abtest_metrics.pkl')

In [23]:
data_metrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,8E4A50179F25185E,2023-09-21,2026-08-08,1052,157,206,1,878,1,2,...,0,0,0,0,0,1,0,1,0,2026-08-17 03:37:29.721348+00:00
1,9250ECA810A81F56,2024-12-28,2026-08-08,588,97,119,1,552,1,3,...,0,0,0,0,0,1,0,0,0,2026-08-17 03:37:29.721348+00:00
2,60F76B6962A5E4F4,2026-05-19,2026-08-08,81,24,20,1,44,1,3,...,0,0,0,0,0,0,0,0,0,2026-08-17 03:37:29.721348+00:00
3,E6EF494B85E74FAE,2022-04-28,2026-08-08,1563,42,43,1,233,1,2,...,0,0,0,0,0,1,0,0,0,2026-08-17 03:37:29.721348+00:00
4,7236846A2EFDD49E,2026-05-27,2026-08-08,73,30,27,1,70,1,1,...,0,0,0,0,0,1,0,0,0,2026-08-17 03:37:29.721348+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3445439,7641CC388B86F103,2022-12-03,2026-08-21,1357,158,207,1,1199,1,6,...,0,0,0,0,0,3,1,1,0,2026-08-28 08:27:32.120637+00:00
3445440,A6C306E84DA50BD8,2025-08-04,2026-08-21,382,149,195,1,377,1,6,...,0,0,0,0,0,6,2,0,0,2026-08-28 08:27:32.120637+00:00
3445441,CD6AD6D77621DDE4,2025-03-27,2026-08-21,512,158,207,1,508,1,10,...,0,0,0,0,0,3,0,1,0,2026-08-28 08:27:32.120637+00:00
3445442,3DD6030932945CCC,2023-09-22,2026-08-21,1064,158,207,1,1054,1,19,...,0,0,0,0,0,4,2,1,0,2026-08-28 08:27:32.120637+00:00


## Merge metrics and assignment

In [24]:
data_metrics = data_metrics.merge(data[['user_id','variant','assigned_dt']], on=['user_id'], how='left')
data_metrics['variant'] = data_metrics['variant'].fillna('NotAssigned')

# A row where dt < assigned_dt is a user's activity from before they were actually randomized
# into a variant - attributing it to Test/control would leak pre-assignment behavior into the
# comparison (assignment can start well before dt, since users get assigned continuously
# throughout the test, not just on day 1). Route those rows back to NotAssigned so
# aggregate_metric drops them the same way it already drops genuinely-unassigned users.
data_metrics.loc[data_metrics['dt'] < data_metrics['assigned_dt'], 'variant'] = 'NotAssigned'

### IAP Rev

In [ ]:
metric = 'iap_rev'
data_metrics_agg = aggregate_metric(data_metrics, metric, 'mean')

plot_metric(
    data_metrics_agg, 
    metric,
    show_overall_diff=True, 
    show_value=True,
    show_rel=True, 
    show_value_cumulative=True,
    show_rel_cumulative=True,
    assignment_start_date='2026-08-05',
    test_start_date='2026-08-17',
    width=1000,
    height=400
)

## Static AB Test Metrics (production snapshot)

Same metrics, but read from `ab_dt_segment_metrics` - the pre-aggregated table [ab_trends.ipynb](../ab_tests_user_cuped_method/ab_trends.ipynb) itself reads from in production, rebuilt daily. Unlike everything above (which always reflects live data, late-arriving events included, up to the moment it's run), this is a static snapshot as of that table's last scheduled refresh (`loading_timestamp`). Comparing the two shows how much a given day's numbers have moved since the last refresh - useful for sanity-checking "why does the dashboard show a different number than my notebook" without re-deriving the whole chain by hand.

In [27]:
query_location = './sql/ab_dt_segment_metrics.sql'
parameters = {
    'experiment_name': 'Temp_Generator',
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1023.4 MB when run.
Estimated query cost: $0.01


In [28]:
data_segment_metrics = pd.DataFrame()

if refresh_data:
    data_segment_metrics = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data_segment_metrics.to_pickle('./data/ab_dt_segment_metrics.pkl')
else:
    data_segment_metrics = pd.read_pickle('./data/ab_dt_segment_metrics.pkl')

In [34]:
data_segment_metrics

,dt,variant,metric,total_users_assigned,dau_assigned,dau,sum_metric,avg_active_metric,sd_active_metric,loading_timestamp
70,2026-07-31,Test,ad_rev,0,0.0,19970.0,1795.115040,0.089891,0.262138,2026-08-25 15:53:35.654456+00:00
393,2026-08-12,Test,ad_rev,42718,31136.0,31136.0,2630.082392,0.084471,0.320015,2026-08-25 15:53:35.654456+00:00
674,2026-08-15,Test,ad_rev,46156,30301.0,30301.0,2422.871186,0.079960,0.286481,2026-08-25 15:53:35.654456+00:00
864,2026-08-10,Test,ad_rev,39333,31446.0,31446.0,2643.387687,0.084061,0.289348,2026-08-25 15:53:35.654456+00:00
986,2026-08-05,Test,ad_rev,112,112.0,112.0,8.124665,0.072542,0.182870,2026-08-25 15:53:35.654456+00:00
...,...,...,...,...,...,...,...,...,...,...
5917,2026-08-16,control,reached_rank1,47130,30555.0,30555.0,1187.000000,0.038848,0.193236,2026-08-25 15:53:35.654456+00:00
6172,2026-08-09,control,reached_rank1,36612,31119.0,31119.0,2235.000000,0.071821,0.258196,2026-08-25 15:53:35.654456+00:00
6446,2026-08-04,control,reached_rank1,0,0.0,29762.0,923.000000,0.031013,0.173355,2026-08-25 15:53:35.654456+00:00
6683,2026-08-08,control,reached_rank1,32669,29739.0,29739.0,573.000000,0.019268,0.137466,2026-08-25 15:53:35.654456+00:00


In [35]:
data_segment_metrics.metric.unique()

array(['ad_rev', 'avg_n_tiles_free', 'coins_earned', 'coins_spent',
       'days_since_install', 'energy_balance_end', 'energy_balance_min',
       'energy_earned', 'energy_earned_ads', 'energy_earned_board',
       'energy_earned_game', 'energy_earned_gems', 'energy_earned_reward',
       'energy_earned_reward_timed_album', 'energy_purchased',
       'energy_spent', 'energy_spent_minus_earned_game',
       'first_n_tiles_free', 'gems_balance_end', 'gems_balance_min',
       'gems_earned', 'gems_earned_board', 'gems_earned_game',
       'gems_earned_reward', 'gems_earned_reward_timed_album',
       'gems_purchased', 'gems_spent', 'gems_spent_bubble',
       'gems_spent_energy', 'gems_spent_gen',
       'gems_spent_minus_earned_game', 'gems_spent_offers',
       'gems_spent_store', 'gross_rev', 'iap_rev', 'iap_rev_dailyoffers',
       'iap_rev_dtc', 'iap_rev_energy', 'iap_rev_gempacks',
       'iap_rev_lops', 'iap_rev_offertrack', 'iap_rev_sale',
       'iap_rev_seasons', 'iap_rev_tipja

In [30]:
metric = 'iap_rev'
data_segment_metrics_agg = aggregate_segment_metric(data_segment_metrics, metric)

plot_metric(
    data_segment_metrics_agg,
    metric,
    show_overall_diff=True,
    show_value=True,
    show_rel=True,
    show_value_cumulative=True,
    show_rel_cumulative=True,
    assignment_start_date='2026-08-05',
    test_start_date='2026-08-17',
    width=1000,
    height=400
)